In [29]:
# Load extension for running R in Jupyter Notebook
%load_ext rpy2.ipython
# Load extension for autoreloading modules
%load_ext autoreload
%autoreload 2

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
import pandas as pd

from utils import (
    functional_richness,
    functional_evenness,
    functional_divergence,
    functional_dispersion,
    raos_Q,
)
from utils import compute_distance_matrix
from utils import compute_relative_abundance
from utils import standardize_trait_matrix

## Testing on a small dataset

In [36]:
traits = pd.DataFrame(
    [[1, 2], [2, 3], [3, 1], [4, 2]],
    columns=["Trait_1", "Trait_2"],
    index=["Sp_0", "Sp_1", "Sp_2", "Sp_3"],
)

abundances = pd.DataFrame(
    [[5, 3, 2, 1], [1, 2, 0, 2]],
    columns=["Sp_0", "Sp_1", "Sp_2", "Sp_3"],
    index=["Plot_A", "Plot_B"],
)

### Python usage of the functional diversity functions

In [39]:
FRic = functional_richness(
    abundances,
    traits,
    calculate_relative_abundance=True,
    standardize_traits_method="z_score",
)

distance_matrix_euclidean = compute_distance_matrix(
    traits, metric="euclidean", standardize_method="z_score"
)
FEve = functional_evenness(
    abundances, distance_matrix=distance_matrix_euclidean, abundance_weighted=True
)

FDiv = functional_divergence(
    abundances,
    traits,
    calculate_relative_abundance=True,
    standardize_traits_method="z_score",
)

FDis = functional_dispersion(
    abundances, traits, abundance_weighted=True, standardize_traits_method="z_score"
)

raos_Q_df = raos_Q(
    abundances,
    distance_matrix=distance_matrix_euclidean,
    calculate_relative_abundance=True,
)

python_results_df = (
    FRic.merge(FEve, on="PID")
    .merge(FDiv, on="PID")
    .merge(FDis, on="PID")
    .merge(raos_Q_df, on="PID")
)
display(python_results_df)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,Plot_A,2.846050,0.734321,0.947928,1.063337,1.264463
1,Plot_B,1.423025,0.989082,0.846568,1.090310,1.224000


In [40]:
r_results_df = None

### R equivalent to calculating the metrics

In [41]:
%%R -i traits,abundances -o r_results_df
library(FD)


trait_mat <- as.matrix(traits)
abun_mat <- as.matrix(abundances)

res <- dbFD(x = trait_mat, a = abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE)

r_results_df <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv,
    R_FDis = res$FDis,
    R_RaoQ = res$RaoQ
)

FRic: No dimensionality reduction was required. The 2 PCoA axes were kept as 'traits'. 


In [42]:
df_merge = python_results_df.merge(r_results_df, on="PID")
df_merge["FRic_ratio"] = df_merge["Functional_Richness"] / df_merge["R_FRic"]
df_merge["FEve_ratio"] = df_merge["Functional_Evenness"] / df_merge["R_FEve"]
df_merge["FDiv_ratio"] = df_merge["Functional_Divergence"] / df_merge["R_FDiv"]
df_merge["FDis_ratio"] = df_merge["Functional_Dispersion"] / df_merge["R_FDis"]
df_merge["RaoQ_ratio"] = df_merge["Raos_Q"] / df_merge["R_RaoQ"]
display(df_merge)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ,FRic_ratio,FEve_ratio,FDiv_ratio,FDis_ratio,RaoQ_ratio
0,Plot_A,2.846050,0.734321,0.947928,1.063337,1.264463,2.846050,0.734321,0.947928,1.063337,1.264463,1.0,1.0,1.0,1.0,1.0
1,Plot_B,1.423025,0.989082,0.846568,1.090310,1.224000,1.423025,0.989082,0.846568,1.090310,1.224000,1.0,1.0,1.0,1.0,1.0


## Testing on the birds dataset

In [43]:
bird_loc = pd.read_csv("./data/example/bird/bird_location.csv")
bird_traits = pd.read_csv("./data/example/bird/bird_traits.csv")

bird_loc = bird_loc.set_index("PID")
bird_traits = bird_traits.set_index("Species")

In [44]:
FRic_bird = functional_richness(
    bird_loc,
    bird_traits,
    calculate_relative_abundance=True,
    standardize_traits_method="z_score",
)


distance_matrix_euclidean_bird = compute_distance_matrix(
    bird_traits, metric="euclidean", standardize_method="z_score"
)
FEve_bird = functional_evenness(
    bird_loc, distance_matrix=distance_matrix_euclidean_bird, abundance_weighted=True
)

FDiv_bird = functional_divergence(
    bird_loc,
    bird_traits,
    calculate_relative_abundance=True,
    standardize_traits_method="z_score",
)

FDis_bird = functional_dispersion(
    bird_loc, bird_traits, abundance_weighted=True, standardize_traits_method="z_score"
)

raos_Q_bird = raos_Q(
    bird_loc,
    distance_matrix=distance_matrix_euclidean_bird,
    calculate_relative_abundance=True,
)

python_results_df_bird = (
    FRic_bird.merge(FEve_bird, on="PID")
    .merge(FDiv_bird, on="PID")
    .merge(FDis_bird, on="PID")
    .merge(raos_Q_bird, on="PID")
)
display(python_results_df_bird)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,elev_250,66.048816,0.656412,0.747405,1.698629,4.574611
1,elev_500,71.465678,0.651052,0.755107,1.737805,4.697285
2,elev_1000,43.354008,0.623858,0.743327,1.564577,3.944307
3,elev_1500,25.466685,0.568285,0.742684,1.471448,3.586842
4,elev_2000,7.725843,0.605025,0.730325,1.244996,2.364448
5,elev_2500,7.046431,0.631438,0.703676,1.265827,2.510058
6,elev_3000,6.749758,0.616293,0.700852,1.331392,2.679338
7,elev_3500,1.427960,0.592614,0.671813,1.345659,2.924699


In [45]:
r_results_df_bird = None

In [46]:
%%R -i bird_loc,bird_traits -o r_results_df_bird
library(FD)
bird_trait_mat <- as.matrix(bird_traits)
bird_abun_mat <- as.matrix(bird_loc)

res <- dbFD(x = bird_trait_mat, a = bird_abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE, print.pco = TRUE)

r_results_df_bird <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv,
    R_FDis = res$FDis,
    R_RaoQ = res$RaoQ
)

FRic: No dimensionality reduction was required. All 4 PCoA axes were kept as 'traits'. 


In [47]:
merge_df = python_results_df_bird.merge(r_results_df_bird, on="PID")

merge_df["FRic_ratio"] = merge_df["Functional_Richness"] / merge_df["R_FRic"]
merge_df["FEve_ratio"] = merge_df["Functional_Evenness"] / merge_df["R_FEve"]
merge_df["FDiv_ratio"] = merge_df["Functional_Divergence"] / merge_df["R_FDiv"]
merge_df["FDis_ratio"] = merge_df["Functional_Dispersion"] / merge_df["R_FDis"]
merge_df["RaoQ_ratio"] = merge_df["Raos_Q"] / merge_df["R_RaoQ"]

display(merge_df)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ,FRic_ratio,FEve_ratio,FDiv_ratio,FDis_ratio,RaoQ_ratio
0,elev_250,66.048816,0.656412,0.747405,1.698629,4.574611,66.048816,0.656412,0.747405,1.698629,4.574611,1.0,1.0,1.0,1.0,1.0
1,elev_500,71.465678,0.651052,0.755107,1.737805,4.697285,71.465678,0.651052,0.755107,1.737805,4.697285,1.0,1.0,1.0,1.0,1.0
2,elev_1000,43.354008,0.623858,0.743327,1.564577,3.944307,43.354008,0.623858,0.743327,1.564577,3.944307,1.0,1.0,1.0,1.0,1.0
3,elev_1500,25.466685,0.568285,0.742684,1.471448,3.586842,25.466685,0.568285,0.742684,1.471448,3.586842,1.0,1.0,1.0,1.0,1.0
4,elev_2000,7.725843,0.605025,0.730325,1.244996,2.364448,7.725843,0.605025,0.730325,1.244996,2.364448,1.0,1.0,1.0,1.0,1.0
5,elev_2500,7.046431,0.631438,0.703676,1.265827,2.510058,7.046431,0.631438,0.703676,1.265827,2.510058,1.0,1.0,1.0,1.0,1.0
6,elev_3000,6.749758,0.616293,0.700852,1.331392,2.679338,6.749758,0.616293,0.700852,1.331392,2.679338,1.0,1.0,1.0,1.0,1.0
7,elev_3500,1.427960,0.592614,0.671813,1.345659,2.924699,1.427960,0.592614,0.671813,1.345659,2.924699,1.0,1.0,1.0,1.0,1.0


## R vs Python Results

After computing the indices in both R and Python, I compared the results by computing the coresponding ratios. 
FEve and FDiv (metrics bounded between 0-1) seem to be consistent.
FRic, FDis, Rao's Q metrics have discrepencies but they are constant across this example. Could be because of how the FD package computes the distance matrix using Gower's distance then uses PCoA in its calculation

## Tree Dataset

Dataset shape:
    Abundance matrix: 20 Sites x 83 Species
    Traits matrix: 54153 Species x 18 Traits

Pre computing the distance matrix for 54153 species is expensive. Can subset to the 83 present species

In [48]:
tree_loc = pd.read_csv("./data/example/trees/tree_location.csv", index_col=0)
tree_traits = pd.read_csv("./data/example/trees/tree_traits.csv", index_col=0)

print("Tree abundances shape: ", tree_loc.shape)
print("Tree traits shape: ", tree_traits.shape)

Tree abundances shape:  (20, 82)
Tree traits shape:  (54153, 18)


In [49]:
print("Number of Tree species in abundances: ", tree_loc.columns.size)
print("Number of Tree species in traits: ", tree_traits.index.size)

print(
    "Number of common Tree species: ",
    len(tree_loc.columns.intersection(tree_traits.index)),
)

tree_traits_sub = tree_traits.loc[tree_traits.index.intersection(tree_loc.columns)]
print("Shape of subsetted trait matrix: ", tree_traits_sub.shape)

Number of Tree species in abundances:  82
Number of Tree species in traits:  54153
Number of common Tree species:  82
Shape of subsetted trait matrix:  (82, 18)


In [50]:
active_species = tree_loc.sum(axis=0) > 0

cleaned_tree_loc = tree_loc.loc[:, active_species]
cleaned_tree_traits = tree_traits_sub.loc[
    tree_traits_sub.index.intersection(cleaned_tree_loc.columns)
]

print("Original tree_loc shape: ", tree_loc.shape)
print("Cleaned tree_loc shape: ", cleaned_tree_loc.shape)

Original tree_loc shape:  (20, 82)
Cleaned tree_loc shape:  (20, 20)


In [ ]:
relative_abundance_tree = compute_relative_abundance(cleaned_tree_loc)

tree_traits_standardized = standardize_trait_matrix(
    cleaned_tree_traits, method="z_score"
)

distance_matrix_euclidean_tree = compute_distance_matrix(
    tree_traits_standardized, metric="euclidean", standardize_method=None
)

In [51]:
FRic_tree = functional_richness(
    relative_abundance_tree,
    tree_traits_standardized,
    calculate_relative_abundance=False,
    standardize_traits_method=None,
)

FEve_tree = functional_evenness(
    relative_abundance_tree,
    distance_matrix=distance_matrix_euclidean_tree,
    abundance_weighted=True,
)

FDiv_tree = functional_divergence(
    relative_abundance_tree,
    tree_traits_standardized,
    calculate_relative_abundance=False,
    standardize_traits_method=None,
)

FDis_tree = functional_dispersion(
    relative_abundance_tree,
    tree_traits_standardized,
    abundance_weighted=True,
    standardize_traits_method=None,
)

raos_Q_tree = raos_Q(
    relative_abundance_tree,
    distance_matrix=distance_matrix_euclidean_tree,
    calculate_relative_abundance=False,
)

python_results_df_tree = (
    FRic_tree.merge(FEve_tree, on="PID")
    .merge(FDiv_tree, on="PID")
    .merge(FDis_tree, on="PID")
    .merge(raos_Q_tree, on="PID")
)
display(python_results_df_tree)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,2_6_89_24_555,NaN,0.693827,NaN,2.491423,8.705807
1,5_6_19_65993_501,NaN,NaN,NaN,NaN,NaN
2,2_6_49_91_555,NaN,NaN,NaN,0.603993,0.923420
3,5_6_107_54130_501,NaN,NaN,NaN,1.556644,3.786158
4,5_6_39_92314_501,NaN,NaN,NaN,0.103211,0.159833
5,2_6_35_91103_501,NaN,NaN,NaN,NaN,NaN
6,3_6_57_85248_501,NaN,NaN,NaN,1.561348,2.452317
7,3_6_55_52_555,NaN,NaN,NaN,3.531530,14.030668
8,4_6_53_88882_501,NaN,NaN,NaN,NaN,NaN
9,2_6_93_69144_501,NaN,NaN,NaN,NaN,NaN


In [52]:
r_results_df_tree = None

In [53]:
%%R -i cleaned_tree_loc,cleaned_tree_traits -o r_results_df_tree
library(FD)
tree_trait_mat <- as.matrix(cleaned_tree_traits)
tree_abun_mat <- as.matrix(cleaned_tree_loc)

res <- dbFD(x = tree_trait_mat, a = tree_abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE, print.pco = TRUE)

r_results_df_tree <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv,
    R_FDis = res$FDis,
    R_RaoQ = res$RaoQ
)

FEVe: Could not be calculated for communities with <3 functionally singular species. 
FDis: Equals 0 in communities with only one functionally singular species. 
FRic: To respect s > t, FRic could not be calculated for communities with <3 functionally singular species. 
FRic: Dimensionality reduction was required. The last 16 PCoA axes (out of 18 in total) were removed. 
FRic: Quality of the reduced-space representation = 0.6213318 
FDiv: Could not be calculated for communities with <3 functionally singular species. 


In [54]:
merge_df_tree = python_results_df_tree.merge(r_results_df_tree, on="PID")

merge_df_tree["FRic_ratio"] = (
    merge_df_tree["Functional_Richness"] / merge_df_tree["R_FRic"]
)
merge_df_tree["FEve_ratio"] = (
    merge_df_tree["Functional_Evenness"] / merge_df_tree["R_FEve"]
)
merge_df_tree["FDiv_ratio"] = (
    merge_df_tree["Functional_Divergence"] / merge_df_tree["R_FDiv"]
)
merge_df_tree["FDis_ratio"] = (
    merge_df_tree["Functional_Dispersion"] / merge_df_tree["R_FDis"]
)
merge_df_tree["RaoQ_ratio"] = merge_df_tree["Raos_Q"] / merge_df_tree["R_RaoQ"]
display(merge_df_tree)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q,R_FRic,R_FEve,R_FDiv,R_FDis,R_RaoQ,FRic_ratio,FEve_ratio,FDiv_ratio,FDis_ratio,RaoQ_ratio
0,2_6_89_24_555,NaN,0.693827,NaN,2.491423,8.705807,7.524440,0.693827,0.548073,2.491423,8.705807,NaN,1.0,NaN,1.0,1.0
1,5_6_19_65993_501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
2,2_6_49_91_555,NaN,NaN,NaN,0.603993,0.923420,NaN,NaN,NaN,0.603993,0.923420,NaN,NaN,NaN,1.0,1.0
3,5_6_107_54130_501,NaN,NaN,NaN,1.556644,3.786158,NaN,NaN,NaN,1.556644,3.786158,NaN,NaN,NaN,1.0,1.0
4,5_6_39_92314_501,NaN,NaN,NaN,0.103211,0.159833,NaN,NaN,NaN,0.103211,0.159833,NaN,NaN,NaN,1.0,1.0
5,2_6_35_91103_501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
6,3_6_57_85248_501,NaN,NaN,NaN,1.561348,2.452317,NaN,NaN,NaN,1.561348,2.452317,NaN,NaN,NaN,1.0,1.0
7,3_6_55_52_555,NaN,NaN,NaN,3.531530,14.030668,NaN,NaN,NaN,3.531530,14.030668,NaN,NaN,NaN,1.0,1.0
8,4_6_53_88882_501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN
9,2_6_93_69144_501,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN


# Species Diversity Metrics

In [47]:
from utils import (
    species_richness,
    shannon_diversity,
    simpsons_index,
    shannon_equitability,
)

In [23]:
bird_loc = pd.read_csv("./data/example/bird/bird_location.csv")
bird_traits = pd.read_csv("./data/example/bird/bird_traits.csv")

bird_loc = bird_loc.set_index("PID")
bird_traits = bird_traits.set_index("Species")

In [49]:
SRichness_bird = species_richness(bird_loc)

Shannon_bird = shannon_diversity(bird_loc)

Simpsons_bird = simpsons_index(bird_loc)

Shannon_equitability_bird = shannon_equitability(bird_loc)

python_results_sdiv_df_bird = (
    SRichness_bird.merge(Shannon_bird, on="PID")
    .merge(Simpsons_bird, on="PID")
    .merge(Shannon_equitability_bird, on="PID")
)
display(python_results_sdiv_df_bird)

,PID,Species Richness,Shannon Diversity,Simpson's Index,Shannon Equitability Index
0,elev_250,125,4.828314,0.992000,1.0
1,elev_500,126,4.836282,0.992063,1.0
2,elev_1000,121,4.795791,0.991736,1.0
3,elev_1500,89,4.488636,0.988764,1.0
4,elev_2000,65,4.174387,0.984615,1.0
5,elev_2500,53,3.970292,0.981132,1.0
6,elev_3000,41,3.713572,0.975610,1.0
7,elev_3500,19,2.944439,0.947368,1.0


In [58]:
r_results_sdiv_df_bird = None

In [59]:
%%R -i bird_loc -o r_results_sdiv_df_bird
library(vegan)
bird_abun_mat <- as.matrix(bird_loc)

H <- diversity(bird_abun_mat, index = "shannon")

S <- specnumber(bird_abun_mat)
pielou_evenness <- H / log(S)

pielou_evenness[is.nan(pielou_evenness)] <- 0

r_results_sdiv_df_bird <- data.frame(
  PID = rownames(bird_abun_mat),
  Shannon_Diversity_R = diversity(bird_abun_mat, "shannon"),
  Simpsons_Index_R = diversity(bird_abun_mat, "simpson"),
  Shannon_Equitability_R = pielou_evenness
)

In [60]:
display(r_results_sdiv_df_bird)

,PID,Shannon_Diversity_R,Simpsons_Index_R,Shannon_Equitability_R
elev_250,elev_250,4.828314,0.992000,1.0
elev_500,elev_500,4.836282,0.992063,1.0
elev_1000,elev_1000,4.795791,0.991736,1.0
elev_1500,elev_1500,4.488636,0.988764,1.0
elev_2000,elev_2000,4.174387,0.984615,1.0
elev_2500,elev_2500,3.970292,0.981132,1.0
elev_3000,elev_3000,3.713572,0.975610,1.0
elev_3500,elev_3500,2.944439,0.947368,1.0


In [57]:
df_merge_sdiv_bird = python_results_sdiv_df_bird.merge(r_results_sdiv_df_bird, on="PID")
df_merge_sdiv_bird["Shannon_Diversity_ratio"] = (
    df_merge_sdiv_bird["Shannon Diversity"] / df_merge_sdiv_bird["Shannon_Diversity_R"]
)
df_merge_sdiv_bird["Simpsons_Index_ratio"] = (
    df_merge_sdiv_bird["Simpson's Index"] / df_merge_sdiv_bird["Simpsons_Index_R"]
)
df_merge_sdiv_bird["Shannon_Equitability_ratio"] = (
    df_merge_sdiv_bird["Shannon Equitability Index"]
    / df_merge_sdiv_bird["Shannon_Equitability_R"]
)
display(df_merge_sdiv_bird)

,PID,Species Richness,Shannon Diversity,Simpson's Index,Shannon Equitability Index,Shannon_Diversity_R,Simpsons_Index_R,Shannon_Equitability_R,Shannon_Diversity_ratio,Simpsons_Index_ratio,Shannon_Equitability_ratio
0,elev_250,125,4.828314,0.992000,1.0,4.828314,0.992000,1.0,1.0,1.0,1.0
1,elev_500,126,4.836282,0.992063,1.0,4.836282,0.992063,1.0,1.0,1.0,1.0
2,elev_1000,121,4.795791,0.991736,1.0,4.795791,0.991736,1.0,1.0,1.0,1.0
3,elev_1500,89,4.488636,0.988764,1.0,4.488636,0.988764,1.0,1.0,1.0,1.0
4,elev_2000,65,4.174387,0.984615,1.0,4.174387,0.984615,1.0,1.0,1.0,1.0
5,elev_2500,53,3.970292,0.981132,1.0,3.970292,0.981132,1.0,1.0,1.0,1.0
6,elev_3000,41,3.713572,0.975610,1.0,3.713572,0.975610,1.0,1.0,1.0,1.0
7,elev_3500,19,2.944439,0.947368,1.0,2.944439,0.947368,1.0,1.0,1.0,1.0


In [61]:
%%R -o r_results_fundiversity
# install.packages("fundiversity")
library(fundiversity)

data(traits_birds)
data(site_sp_birds)
scaled_traits <- as.data.frame(scale(traits_birds))
r_results_fundiversity <- fd_fdis(scaled_traits, site_sp_birds)

In [60]:
python_results_df_bird

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,Functional_Dispersion,Raos_Q
0,elev_250,66.048816,0.656412,0.747405,1.698629,4.574611
1,elev_500,71.465678,0.651052,0.755107,1.737805,4.697285
2,elev_1000,43.354008,0.623858,0.743327,1.564577,3.944307
3,elev_1500,25.466685,0.568285,0.742684,1.471448,3.586842
4,elev_2000,7.725843,0.605025,0.730325,1.244996,2.364448
5,elev_2500,7.046431,0.631438,0.703676,1.265827,2.510058
6,elev_3000,6.749758,0.616293,0.700852,1.331392,2.679338
7,elev_3500,1.427960,0.592614,0.671813,1.345659,2.924699
